In [15]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict, Annotated
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage,HumanMessage
from dotenv import load_dotenv
from langgraph.checkpoint.memory import MemorySaver

In [10]:
load_dotenv()

True

In [3]:
from langgraph.graph.message import add_messages
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage],add_messages]

In [11]:
llm = ChatOpenAI(model="gpt-4o-mini")
def chat_node(state:ChatState):

    messages = state['messages']

    response = llm.invoke(messages)

    return {'messages':[response]}




In [16]:
checkpointer = MemorySaver()
graph = StateGraph(ChatState)

graph.add_node('chat_node',chat_node)

graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)

chatbot=graph.compile(checkpointer=checkpointer)

In [12]:
initial_state ={
    'messages' :[HumanMessage(content = 'What is the distance of Jaipur to AmritSar?')]
}

chatbot.invoke(initial_state)['messages'][-1]

AIMessage(content='The distance from Jaipur to Amritsar is approximately 500 kilometers (about 310 miles) by road. The actual distance can vary slightly depending on the specific route taken. If you are considering traveling by train or air, the distances may differ as well.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 18, 'total_tokens': 70, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_2247163164', 'id': 'chatcmpl-E45iyvhZBjyv32FfaNXeQ4LHbJFmc', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f8512-b10e-7f00-9666-1183e5f7159e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 18, 'output_tokens': 52, 'tot

In [17]:
thread_id = '1'
while True:

    user_message = input('Type here : ')
    print('user: ',user_message)

    if user_message.strip().lower() in ['exit','quit','bye']:
        break
    config = {'configurable':{'thread_id': thread_id}}
    response = chatbot.invoke({'messages':[HumanMessage(content=user_message)]},config=config)

    print('AI : ',response['messages'][-1].content)

user:  hi my name is vansh 
AI :  Hi Vansh! How can I assist you today?
user:  can you tell me my name 
AI :  Your name is Vansh! How can I help you further?
user:  exit 


Now what happend here was , we simply made a chatbot and looped it until the user types exit , quit or bye .. now the chatbot was giving the answers clearly using an llm , but the major problem was that if i start the chat and i say that , hi my name is vansh ...then chatbot answers that hi vansh, how may i assist you today ... then when i ask the chatbot that what is my name it will tell that it does not have any context about my name ..... thats a major flaw 
Now why it is happening 
when user starts chatting , it is sending human message which is going to llm and then according to workflow it is simply going to the END state , and at next iteration it is re invoking without having any previous conversation 
that is why its memory is empty ... so for having atleast a conversation history , we used persistace , which will be covered in next notebook in detail... ,with that we are defining a thread id , for a particular user 
 

Till now chatbot is working but it is not storing the memory 
so we'll use persistance
